In [11]:
import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
from astropy.time import Time
from astropy.coordinates import TEME, ITRS, EarthLocation, CartesianRepresentation
import astropy.units as u
from datetime import datetime, timezone
from sgp4.api import Satrec



# Login + Session

In [12]:
ST_BASE    = "https://www.space-track.org"
ST_LOGIN   = f"{ST_BASE}/ajaxauth/login"
ST_QUERY   = f"{ST_BASE}/basicspacedata/query"

USERNAME = "nathandarby2022@gmail.com"   # replace with your Space-Track credentials
PASSWORD = "Boobear32*Boobear32"             # consider loading from env variable

session = requests.Session()

resp = session.post(ST_LOGIN, data={"identity": "nathandarby2022@gmail.com", "password": "Boobear32*Boobear32"})

if resp.status_code == 200:
    print("Logged in to Space-Track successfully!")
else:
    print(f"Login failed: {resp.status_code}")

Logged in to Space-Track successfully!


In [13]:
resp = session.post(ST_LOGIN, data={"identity": "nathandarby2022@gmail.com", "password": "Boobear32*Boobear32"})
print(resp.status_code)

200


# SATCAT Fetch + Cache

In [14]:
import json
import os
from datetime import datetime, timezone

SATCAT_CACHE_FILE = os.path.expanduser("~/satcat_cache.json")

def get_norad_ids(intldes, session, force=False):
    """
    Get all NORAD IDs for a launch group.
    Reads from local cache if available, otherwise queries Space-Track satcat.
    """
    # Check cache first
    if os.path.exists(SATCAT_CACHE_FILE) and not force:
        with open(SATCAT_CACHE_FILE) as f:
            cache = json.load(f)
        
        if cache.get('intldes') == intldes:
            print(f"Loaded {len(cache['norad_ids'])} NORAD IDs from cache")
            print(f"   Launch:    {intldes}")
            print(f"   Cached at: {cache['fetched_at']}")
            for obj in cache['objects']:
                print(f"   {obj['NORAD_CAT_ID']} — {obj['SATNAME']}")
            return cache['norad_ids']
        else:
            print(f"Cache is for different launch ({cache.get('intldes')}) — fetching fresh")

    # Query satcat
    print(f"Querying Space-Track satcat for launch {intldes}...")
    url = (f"{ST_QUERY}/class/satcat"
           f"/INTLDES/{intldes}~~"
           f"/OBJECT_TYPE/Payload"
           f"/orderby/NORAD_CAT_ID"
           f"/format/json")
    
    resp = session.get(url)
    
    if resp.status_code != 200:
        print(f"satcat query failed: {resp.status_code}")
        return []
    
    objects = resp.json()
    norad_ids = [obj['NORAD_CAT_ID'] for obj in objects]
    
    # Save to cache
    cache = {
        'intldes':    intldes,
        'fetched_at': datetime.now(timezone.utc).isoformat(),
        'norad_ids':  norad_ids,
        'objects':    [{'NORAD_CAT_ID': obj['NORAD_CAT_ID'], 
                        'SATNAME':      obj['SATNAME']} for obj in objects]
    }
    with open(SATCAT_CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)
    
    print(f"Fetched and cached {len(norad_ids)} payloads for launch {intldes}")
    for obj in objects:
        print(f"   {obj['NORAD_CAT_ID']} — {obj['SATNAME']}")
    
    return norad_ids

# Current TLE Fetch + Cache

In [15]:
TLE_CACHE_FILE = os.path.expanduser("~/tle_cache.json")

def fetch_tles(norad_ids, session, chunk_size=50, force=False):
    if os.path.exists(TLE_CACHE_FILE) and not force:
        with open(TLE_CACHE_FILE) as f:
            cache = json.load(f)
        fetched_at = datetime.fromisoformat(cache['fetched_at'])
        age = datetime.now(timezone.utc) - fetched_at
        if age.total_seconds() < 3600:
            remaining = 3600 - age.total_seconds()
            print(f"Loaded TLEs from Cache")
            print(f"   Last Query:            {int(age.total_seconds()//60)} min ago")
            print(f"   Next Available Query:  {int(remaining//60)} min")
            print(f"   Total TLEs:            {len(cache['tles'])}")
            return cache['tles']
        else:
            print("Cache expired — fetching fresh TLEs")

    all_tles = []
    for i in range(0, len(norad_ids), chunk_size):
        chunk = norad_ids[i:i+chunk_size]
        ids_str = ','.join(str(n) for n in chunk)
        url = (f"{ST_QUERY}/class/gp"
               f"/NORAD_CAT_ID/{ids_str}"
               f"/decay_date/null-val"
               f"/orderby/NORAD_CAT_ID"
               f"/format/json")
        resp = session.get(url)
        print(f"   Chunk {i//chunk_size + 1}: status {resp.status_code}")
        if resp.status_code != 200 or not resp.text.strip():
            continue
        for obj in resp.json():
            if obj.get('TLE_LINE1') and obj.get('TLE_LINE2'):
                all_tles.append({
                    'name':  obj['OBJECT_NAME'],
                    'line1': obj['TLE_LINE1'],
                    'line2': obj['TLE_LINE2']
                })

    # Save to cache
    cache = {'fetched_at': datetime.now(timezone.utc).isoformat(), 'tles': all_tles}
    with open(TLE_CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)

    print(f"\nLoaded TLEs from Query")
    print(f"   Last Query:            just now")
    print(f"   Next Available Query:  60 min")
    print(f"   Total TLEs:            {len(all_tles)}")
    return all_tles

In [16]:
# Transporter-9 — Oct 2023
norad_ids = get_norad_ids('2023-174', session, force=False)
tles = fetch_tles(norad_ids, session, force=False)

Loaded 95 NORAD IDs from cache
   Launch:    2023-174
   Cached at: 2026-04-21T03:54:00.131847+00:00
   58256 — CONNECTA T3.1
   58257 — MANTIS
   58258 — ORBASTRO-PC-1
   58259 — LEMUR 2 BASS
   58260 — LEMUR 2 DILIGHTFUL
   58262 — PLATERO
   58263 — LEMUR 2 GOOD-VIBES
   58264 — BRO-11
   58266 — DJIBOUTI-1A
   58267 — LEMUR 2 SANITA-VETRA
   58268 — CONNECTA T3.2
   58269 — IRIS-C2
   58270 — FLOCK 4Q 1
   58271 — FLOCK 4Q 16
   58272 — TIGER-6
   58273 — FLOCK 4Q 11
   58274 — FLOCK 4Q 36
   58275 — FLOCK 4Q 12
   58276 — GENMAT-1
   58277 — TIGER-5
   58278 — FLOCK 4Q 19
   58279 — FLOCK 4Q 8
   58280 — FLOCK 4Q 28
   58281 — AETHER-1
   58282 — FLOCK 4Q 27
   58283 — FLOCK 4Q 17
   58284 — FLOCK 4Q 26
   58285 — FLOCK 4Q 31
   58286 — FLOCK 4Q 30
   58287 — PICO-01B009
   58288 — ICEYE-X31
   58291 — FALCONSAT-10
   58292 — UMBRA-08
   58293 — ICEYE-X32
   58294 — ICEYE-X34
   58295 — SPACEVAN-001
   58296 — PELICAN-1 3001
   58297 — UMBRA-07
   58298 — SPIP
   58299 — AETHER-2


# Dataframe

In [17]:
rows = []
for t in tles:
    line1 = t['line1']
    intldes_raw = line1[9:17].strip()
    year  = intldes_raw[:2]
    num   = intldes_raw[2:5]
    piece = intldes_raw[5:]
    full_year = f"19{year}" if int(year) >= 57 else f"20{year}"
    intldes = f"{full_year}-{num}{piece}"
    
    rows.append({
        'International Designator': intldes,
        'NORAD Catalog Number':     line1[2:7].strip(),
        'Name':                     t['name'],
        'TLE Line 1':               line1,
        'TLE Line 2':               t['line2']
    })

df_tles = pd.DataFrame(rows)
print(f"Total objects: {len(df_tles)}")
df_tles

Total objects: 54


,International Designator,NORAD Catalog Number,Name,TLE Line 1,TLE Line 2
0,2023-174C,58258,ORBASTRO-PC-1,1 58258U 23174C 26112.58394282 .00017790 0...,2 58258 97.3771 197.5336 0006182 24.1106 336...
1,2023-174D,58259,LEMUR 2 BASS,1 58259U 23174D 26112.14836421 .00012816 0...,2 58259 97.3839 196.1594 0006897 24.4521 335...
2,2023-174G,58262,PLATERO,1 58262U 23174G 26112.22123035 .00023665 0...,2 58262 97.3913 207.8417 0003428 312.8437 47...
3,2023-174J,58264,BRO-11,1 58264U 23174J 26112.61499275 .00004129 0...,2 58264 97.3910 189.9895 0011497 51.2882 308...
4,2023-174Q,58270,FLOCK 4Q 1,1 58270U 23174Q 26112.51185857 .00065917 0...,2 58270 97.3813 203.9529 0003805 342.2641 17...
5,2023-174R,58271,FLOCK 4Q 16,1 58271U 23174R 26112.18883909 .00016403 0...,2 58271 97.3817 199.1080 0005583 18.7679 341...
6,2023-174S,58272,TIGER-6,1 58272U 23174S 26112.51236723 .00154352 0...,2 58272 97.3863 208.4976 0003699 263.4176 96...
7,2023-174T,58273,FLOCK 4Q 11,1 58273U 23174T 26112.50777388 .00015382 0...,2 58273 97.3867 199.9669 0005210 15.6896 344...
8,2023-174U,58274,FLOCK 4Q 36,1 58274U 23174U 26112.50033179 .00023938 0...,2 58274 97.3845 203.0302 0005067 347.2203 12...
9,2023-174V,58275,FLOCK 4Q 12,1 58275U 23174V 26112.21667242 .00077342 0...,2 58275 97.3832 202.8588 0002660 307.3057 52...


# Historical TLE fetch (gp_history) (!!! Careful !!!)

In [35]:
import os
os.remove(os.path.expanduser("~/tle_history_cache.json"))
print("Cache cleared")

Cache cleared


In [36]:
# Section 5 — Historical TLE Fetch (gp_history)
# One-time query — never re-query, always load from cache

HISTORY_CACHE_FILE = os.path.expanduser("~/tle_history_cache.json")

with open(SATCAT_CACHE_FILE) as f:
    satcat_cache = json.load(f)

def fetch_historical_tles(norad_ids, start_date, end_date, session, chunk_size=50):
    """
    Fetch historical TLEs from Space-Track gp_history class.
    One-time query only — results cached permanently to disk.
    start_date, end_date: 'YYYY-MM-DD' strings
    """
    # Check cache — never re-query gp_history
    if os.path.exists(HISTORY_CACHE_FILE):
        with open(HISTORY_CACHE_FILE) as f:
            cache = json.load(f)
        print(f"Historical TLEs loaded from cache")
        print(f"   Launch:     {cache['intldes']}")
        print(f"   Epoch range: {cache['start_date']} → {cache['end_date']}")
        print(f"   Records:    {len(cache['tles'])}")
        return cache['tles']

    print(f"Querying gp_history: {start_date} → {end_date}")
    print(f"WARNING: This is a one-time query — results will be cached permanently")

    all_tles = []
    for i in range(0, len(norad_ids), chunk_size):
        chunk   = norad_ids[i:i+chunk_size]
        ids_str = ','.join(str(n) for n in chunk)

        url = (f"{ST_QUERY}/class/gp_history"
               f"/NORAD_CAT_ID/{ids_str}"
               f"/epoch/{start_date}--{end_date}"
               f"/orderby/NORAD_CAT_ID,EPOCH"
               f"/format/json")

        resp = session.get(url)
        print(f"   Chunk {i//chunk_size + 1}: status {resp.status_code} — {len(resp.json()) if resp.status_code == 200 else 0} records")

        if resp.status_code != 200 or not resp.text.strip():
            continue

        for obj in resp.json():
            if obj.get('TLE_LINE1') and obj.get('TLE_LINE2'):
                all_tles.append({
                    'name':  obj['OBJECT_NAME'],
                    'norad': obj['NORAD_CAT_ID'],
                    'epoch': obj['EPOCH'],
                    'line1': obj['TLE_LINE1'],
                    'line2': obj['TLE_LINE2']
                })

    # Save to cache permanently
    cache = {
        'intldes':    satcat_cache['intldes'],
        'start_date': start_date,
        'end_date':   end_date,
        'fetched_at': datetime.now(timezone.utc).isoformat(),
        'tles':       all_tles
    }
    with open(HISTORY_CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)

    print(f"\nHistorical TLEs fetched and cached permanently")
    print(f"   Total records: {len(all_tles)}")
    print(f"   Cache file:    {HISTORY_CACHE_FILE}")
    return all_tles

# Transporter-9 launched October 6, 2023
# Query first 7 days post-deployment
historical_tles = fetch_historical_tles(
    norad_ids,
    start_date='2023-11-11',
    end_date='2024-01-11',
    session=session
)

Querying gp_history: 2023-11-11 → 2024-01-11
   Chunk 1: status 200 — 5755 records
   Chunk 2: status 200 — 4430 records

Historical TLEs fetched and cached permanently
   Total records: 10185
   Cache file:    /Users/nathandarby/tle_history_cache.json


# Fetched TLE time to identification analysis (useful for pitches)

In [30]:
first_catalog_clean = first_catalog.drop_duplicates(subset='norad').sort_values('days_after_launch')
print(f"Unique satellites: {len(first_catalog_clean)}")
print(f"Mean days to catalog: {first_catalog_clean['days_after_launch'].mean():.1f}")
print(f"Range: {first_catalog_clean['days_after_launch'].min():.1f} — {first_catalog_clean['days_after_launch'].max():.1f} days")

Unique satellites: 80
Mean days to catalog: 17.6
Range: 17.1 — 17.7 days


In [31]:


# Check first cataloged date per satellite from historical TLEs
hist_df = pd.DataFrame(historical_tles)
hist_df['epoch'] = pd.to_datetime(hist_df['epoch'], utc=True)

first_catalog = hist_df.groupby(['norad', 'name'])['epoch'].min().reset_index()
first_catalog.columns = ['norad', 'name', 'first_cataloged']
first_catalog['days_after_launch'] = (first_catalog['first_cataloged'] - pd.Timestamp('2023-11-11', tz='UTC')).dt.total_seconds() / 86400
first_catalog = first_catalog.sort_values('days_after_launch').reset_index(drop=True)

print(f"{'Name':<25} {'NORAD':<8} {'First Cataloged':<25} {'Days After Launch':>17}")
print("-" * 80)
for _, row in first_catalog.iterrows():
    print(f"{row['name']:<25} {row['norad']:<8} {str(row['first_cataloged']):<25} {row['days_after_launch']:>17.1f}")

print(f"\nSummary:")
print(f"   Earliest cataloged: {first_catalog['days_after_launch'].min():.1f} days — {first_catalog.iloc[0]['name']}")
print(f"   Latest cataloged:   {first_catalog['days_after_launch'].max():.1f} days — {first_catalog.iloc[-1]['name']}")
print(f"   Mean:               {first_catalog['days_after_launch'].mean():.1f} days")

Name                      NORAD    First Cataloged           Days After Launch
--------------------------------------------------------------------------------
TBA - TO BE ASSIGNED      58323    2023-11-28 03:24:35.342208+00:00              17.1
OBJECT BV                 58323    2023-11-28 03:24:35.342208+00:00              17.1
TBA - TO BE ASSIGNED      58268    2023-11-28 03:44:45.962592+00:00              17.2
OBJECT N                  58268    2023-11-28 03:44:45.962592+00:00              17.2
TBA - TO BE ASSIGNED      58333    2023-11-28 04:53:33.113472+00:00              17.2
OBJECT CF                 58333    2023-11-28 04:53:33.113472+00:00              17.2
TBA - TO BE ASSIGNED      58325    2023-11-28 04:58:52.652640+00:00              17.2
OBJECT BX                 58325    2023-11-28 04:58:52.652640+00:00              17.2
TBA - TO BE ASSIGNED      58338    2023-11-28 14:21:20.089152+00:00              17.6
OBJECT CL                 58338    2023-11-28 14:21:20.089152+00:0

In [33]:
# Get TBA/OBJECT name and confirmed name separately
tba_names = hist_df[~hist_df['is_confirmed']].groupby('norad')['name'].first().reset_index()
tba_names.columns = ['norad', 'unconfirmed_name']

confirmed_names = hist_df[hist_df['is_confirmed']].groupby('norad')['name'].first().reset_index()
confirmed_names.columns = ['norad', 'confirmed_name']

# Merge everything
confirmation_df = first_cataloged.merge(first_confirmed, on='norad', how='left')
confirmation_df = confirmation_df.merge(tba_names, on='norad', how='left')
confirmation_df = confirmation_df.merge(confirmed_names, on='norad', how='left')
confirmation_df['days_to_catalog']  = (confirmation_df['first_cataloged'] - pd.Timestamp('2023-11-11', tz='UTC')).dt.total_seconds() / 86400
confirmation_df['days_to_confirm']  = (confirmation_df['first_confirmed'] - pd.Timestamp('2023-11-11', tz='UTC')).dt.total_seconds() / 86400
confirmation_df['confirmation_lag'] = confirmation_df['days_to_confirm'] - confirmation_df['days_to_catalog']
confirmation_df = confirmation_df.sort_values('days_to_confirm').reset_index(drop=True)

print(f"{'Unconfirmed Name':<25} {'Confirmed Name':<25} {'NORAD':<8} {'Cataloged':>10} {'Confirmed':>10} {'Lag':>8}")
print("-" * 95)
for _, row in confirmation_df.iterrows():
    confirmed     = f"{row['days_to_confirm']:.1f}"  if not pd.isna(row['days_to_confirm'])  else "NOT YET"
    lag           = f"{row['confirmation_lag']:.1f}" if not pd.isna(row['confirmation_lag']) else "N/A"
    confirmed_name = row['confirmed_name'] if not pd.isna(row['confirmed_name']) else "UNCONFIRMED"
    print(f"{row['unconfirmed_name']:<25} {confirmed_name:<25} {row['norad']:<8} {row['days_to_catalog']:>10.1f} {confirmed:>10} {lag:>8}")

print(f"\nSummary:")
print(f"   Mean days to catalog:      {confirmation_df['days_to_catalog'].mean():.1f} days")
print(f"   Mean days to confirm:      {confirmation_df['days_to_confirm'].mean():.1f} days")
print(f"   Mean confirmation lag:     {confirmation_df['confirmation_lag'].mean():.1f} days")
print(f"   Unconfirmed in 30 days:    {confirmation_df['confirmed_name'].isna().sum()}")

Unconfirmed Name          Confirmed Name            NORAD     Cataloged  Confirmed      Lag
-----------------------------------------------------------------------------------------------
TBA - TO BE ASSIGNED      FALCONSAT-10              58291          17.7       17.7      0.0
TBA - TO BE ASSIGNED      ICEYE-X31                 58288          17.7       18.8      1.1
OBJECT AV                 AETHER-2                  58299          17.7       18.9      1.2
OBJECT AX                 IMPULSE-1 MIRA            58301          17.7       18.9      1.2
TBA - TO BE ASSIGNED      SPIP                      58298          17.7       18.9      1.2
OBJECT AQ                 ICEYE-34                  58294          17.7       18.9      1.2
OBJECT AP                 ICEYE-X32                 58293          17.7       19.8      2.1
OBJECT AB                 AETHER-1                  58281          17.7       19.8      2.1
TBA - TO BE ASSIGNED      ICEYE-X35                 58302          17.7     

In [34]:
# Check if history cache covers month 2, if not we need to re-query
# First check what we already have
print(f"Current history range: {hist_df['epoch'].min()} → {hist_df['epoch'].max()}")
print(f"Records: {len(hist_df)}")

Current history range: 2023-11-28 03:24:35.342208+00:00 → 2023-12-10 21:51:49.951584+00:00
Records: 2281


In [37]:
# Reload cache
with open(HISTORY_CACHE_FILE) as f:
    history_cache = json.load(f)

historical_tles = history_cache['tles']
hist_df = pd.DataFrame(historical_tles)
hist_df['epoch'] = pd.to_datetime(hist_df['epoch'], utc=True)
hist_df['is_confirmed'] = hist_df['name'].apply(is_confirmed)

# Rerun confirmation analysis
tba_names       = hist_df[~hist_df['is_confirmed']].groupby('norad')['name'].first().reset_index()
tba_names.columns = ['norad', 'unconfirmed_name']

confirmed_names = hist_df[hist_df['is_confirmed']].groupby('norad')['name'].first().reset_index()
confirmed_names.columns = ['norad', 'confirmed_name']

first_cataloged = hist_df.groupby('norad')['epoch'].min().reset_index()
first_cataloged.columns = ['norad', 'first_cataloged']

first_confirmed = hist_df[hist_df['is_confirmed']].groupby('norad')['epoch'].min().reset_index()
first_confirmed.columns = ['norad', 'first_confirmed']

confirmation_df = first_cataloged.merge(first_confirmed, on='norad', how='left')
confirmation_df = confirmation_df.merge(tba_names, on='norad', how='left')
confirmation_df = confirmation_df.merge(confirmed_names, on='norad', how='left')
confirmation_df['days_to_catalog']  = (confirmation_df['first_cataloged'] - pd.Timestamp('2023-11-11', tz='UTC')).dt.total_seconds() / 86400
confirmation_df['days_to_confirm']  = (confirmation_df['first_confirmed'] - pd.Timestamp('2023-11-11', tz='UTC')).dt.total_seconds() / 86400
confirmation_df['confirmation_lag'] = confirmation_df['days_to_confirm'] - confirmation_df['days_to_catalog']
confirmation_df = confirmation_df.sort_values('days_to_confirm').reset_index(drop=True)

print(f"{'Unconfirmed Name':<25} {'Confirmed Name':<25} {'NORAD':<8} {'Cataloged':>10} {'Confirmed':>10} {'Lag':>8}")
print("-" * 95)
for _, row in confirmation_df.iterrows():
    confirmed      = f"{row['days_to_confirm']:.1f}"  if not pd.isna(row['days_to_confirm'])  else "NOT YET"
    lag            = f"{row['confirmation_lag']:.1f}" if not pd.isna(row['confirmation_lag']) else "N/A"
    confirmed_name = row['confirmed_name'] if not pd.isna(row['confirmed_name']) else "UNCONFIRMED"
    print(f"{row['unconfirmed_name']:<25} {confirmed_name:<25} {row['norad']:<8} {row['days_to_catalog']:>10.1f} {confirmed:>10} {lag:>8}")

print(f"\nSummary:")
print(f"   Total unique objects:      {len(confirmation_df)}")
print(f"   Mean days to catalog:      {confirmation_df['days_to_catalog'].mean():.1f} days")
print(f"   Mean days to confirm:      {confirmation_df['days_to_confirm'].mean():.1f} days")
print(f"   Mean confirmation lag:     {confirmation_df['confirmation_lag'].mean():.1f} days")
print(f"   Confirmed in 60 days:      {confirmation_df['confirmed_name'].notna().sum()}")
print(f"   Unconfirmed in 60 days:    {confirmation_df['confirmed_name'].isna().sum()}")

Unconfirmed Name          Confirmed Name            NORAD     Cataloged  Confirmed      Lag
-----------------------------------------------------------------------------------------------
TBA - TO BE ASSIGNED      FALCONSAT-10              58291          17.7       17.7      0.0
TBA - TO BE ASSIGNED      ICEYE-X31                 58288          17.7       18.8      1.1
OBJECT AV                 AETHER-2                  58299          17.7       18.9      1.2
OBJECT AX                 IMPULSE-1 MIRA            58301          17.7       18.9      1.2
TBA - TO BE ASSIGNED      SPIP                      58298          17.7       18.9      1.2
OBJECT AQ                 ICEYE-34                  58294          17.7       18.9      1.2
OBJECT AP                 ICEYE-X32                 58293          17.7       19.8      2.1
OBJECT AB                 AETHER-1                  58281          17.7       19.8      2.1
TBA - TO BE ASSIGNED      ICEYE-X35                 58302          17.7     

In [38]:
unconfirmed = confirmation_df[confirmation_df['confirmed_name'].isna()]
print(f"Still unconfirmed after 60 days:")
for _, row in unconfirmed.iterrows():
    print(f"   {row['norad']} — {row['unconfirmed_name']}")

Still unconfirmed after 60 days:
   58258 — TBA - TO BE ASSIGNED
   58266 — TBA - TO BE ASSIGNED
   58269 — TBA - TO BE ASSIGNED
   58287 — TBA - TO BE ASSIGNED
   58326 — TBA - TO BE ASSIGNED
   58330 — TBA - TO BE ASSIGNED
   58339 — TBA - TO BE ASSIGNED
   58343 — TBA - TO BE ASSIGNED
   58344 — TBA - TO BE ASSIGNED
   58345 — TBA - TO BE ASSIGNED
   58563 — TBA - TO BE ASSIGNED
   58564 — TBA - TO BE ASSIGNED
   58566 — TBA - TO BE ASSIGNED
   58572 — TBA - TO BE ASSIGNED
   58690 — TBA - TO BE ASSIGNED
   58697 — TBA - TO BE ASSIGNED


In [39]:
unconfirmed_norads = ['58258', '58266', '58269', '58287', '58326', '58330', 
                       '58339', '58343', '58344', '58345', '58563', '58564', 
                       '58566', '58572', '58690', '58697']

# Check against current TLE dataframe
print("Unconfirmed objects — current names:")
print(f"{'NORAD':<8} {'Current Name':<30}")
print("-" * 40)
for norad in unconfirmed_norads:
    match = df_tles[df_tles['NORAD Catalog Number'] == norad]
    if len(match) > 0:
        print(f"{norad:<8} {match.iloc[0]['Name']:<30}")
    else:
        print(f"{norad:<8} NOT IN CURRENT CATALOG")

Unconfirmed objects — current names:
NORAD    Current Name                  
----------------------------------------
58258    ORBASTRO-PC-1                 
58266    NOT IN CURRENT CATALOG
58269    NOT IN CURRENT CATALOG
58287    PICO-01B009                   
58326    PICO-01A003                   
58330    NOT IN CURRENT CATALOG
58339    NOT IN CURRENT CATALOG
58343    NOT IN CURRENT CATALOG
58344    NOT IN CURRENT CATALOG
58345    NOT IN CURRENT CATALOG
58563    NOT IN CURRENT CATALOG
58564    NOT IN CURRENT CATALOG
58566    NOT IN CURRENT CATALOG
58572    NOT IN CURRENT CATALOG
58690    NOT IN CURRENT CATALOG
58697    NOT IN CURRENT CATALOG


In [40]:
# Query satcat for decay dates of unconfirmed objects
unconfirmed_gone = ['58266', '58269', '58330', '58339', '58343', 
                     '58344', '58345', '58563', '58564', '58566', 
                     '58572', '58690', '58697']

ids_str = ','.join(unconfirmed_gone)
url = f"{ST_QUERY}/class/satcat/NORAD_CAT_ID/{ids_str}/format/json"
resp = session.get(url)
for obj in resp.json():
    print(f"{obj['NORAD_CAT_ID']:<8} {obj['SATNAME']:<30} Decay: {obj['DECAY']}")

58266    DJIBOUTI-1A                    Decay: 2025-09-02
58269    IRIS-C2                        Decay: 2025-08-17
58330    PICO-01G005                    Decay: 2025-05-06
58339    PLATFORM-5                     Decay: 2025-04-22
58343    PICO-01H004                    Decay: 2024-06-26
58344    PICO-01C008                    Decay: 2024-07-23
58345    PICO-01I002                    Decay: 2024-06-26
58563    PICO-01F006                    Decay: 2024-06-27
58564    PICO-01E001                    Decay: 2024-06-25
58566    UNICORN-2J                     Decay: 2024-06-27
58572    TIME WE'LL TELL                Decay: 2024-12-03
58690    PICO-01D007                    Decay: 2024-12-31
58697    LEMUR 2 NANAZ                  Decay: 2025-03-08
